# Fine-tune Amazon Nova on Amazon Bedrock for Document to JSON

In [ ]:
import boto3

In [ ]:
%pip install pypdfium2==4.30.1 pandas==2.2.3 huggingface_hub[hf_transfer]==0.27.1 datasets==3.2.0 ipywidgets==8.1.5 tqdm==4.67.1 genson boto3==1.38.10 sagemaker --quiet 

## Preparing Data for Fine-tuning Amazon Nova Understandin Models

In [ ]:
import os
# Configure environment for optimal performance
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # Enable fast transfers
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disable tokenizer warnings

In [ ]:
from huggingface_hub import notebook_login
import sagemaker
import boto3

# Authenticate with Hugging Face Hub for private datasets
# notebook_login()  # Uncomment for private datasets

region="us-east-1" # us-east-1 for Amazon Nova models
# boto_session = boto3.Session(region_name=region)
boto_session = boto3.Session(
    region_name=region
)

# Initialize AWS resources
session = sagemaker.Session(boto_session=boto_session)
default_bucket_name = "nova-doc-to-json-w2" #session.default_bucket()
dataset_s3_prefix = "w2-train-data-amazon-nova" # "fatura2-train-data-amazon-nova"
dataset_s3_uri = f"s3://{default_bucket_name}/{dataset_s3_prefix}/"

data_main_dir = "./data/"
hf_dataset_name = "singhsays/fake-w2-us-tax-form-dataset"
dataset_dir = os.path.join(data_main_dir, hf_dataset_name.split("/")[-1])
os.makedirs(dataset_dir, exist_ok=True)

In [ ]:
account_id = session.account_id()

In [ ]:
dataset_s3_uri

In [ ]:
import requests
import zipfile
from tqdm import tqdm
from datasets import load_dataset

# Create directories to store images and metadata
os.makedirs('ocr_images/train', exist_ok=True)
os.makedirs('ocr_images/val', exist_ok=True)
os.makedirs('ocr_images/test', exist_ok=True)


# Select a subset for our fine-tuning task
# We want 1200 examples total (1000 train, 100 val, 100 test)
dataset = load_dataset(hf_dataset_name)

dataset

In [ ]:
# [Optional] reduce the dataset size to 300 samples for faster training and already good enough results
# Note do not change the test dataset size so that you can compare the evaluation
# dataset["train"] = dataset["train"].train_test_split(300)["test"]
# dataset["dev"] = dataset["dev"].train_test_split(75)["test"]

In [ ]:
target_column = "ground_truth" # "target_data"

In [ ]:
from IPython.display import JSON
import json

JSON(json.loads(dataset["test"].to_pandas()[target_column].iloc[1]))

In [ ]:
def custom_flatten_function(example):
    """
    Custom function to flatten specific JSON structure
    """
    # Check if the column exists and contains gt_parse

    if target_column in example:
        data = example[target_column]
        
        if isinstance(data, str):
            # If it's a JSON string, parse it first
            data = json.loads(data)


        if 'gt_parse' in data:
            # Move gt_parse contents to root
            gt_parse_content = data['gt_parse']
            return {target_column: json.dumps(gt_parse_content)}
            
    return example

# Apply to all splits
dataset = dataset.map(custom_flatten_function)

## 4. Document Processing Pipeline

### Key Processing Stages:
1. **PDF Rendering**: Convert PDF pages to PNG images at 153% scale for OCR optimization
2. **Image Normalization**: Standardize image formats and orientations
3. **Bounding Box Transformation**: Convert absolute coordinates to Swift XML format
4. **Hierarchy Flattening**: Simplify nested document structures while preserving relationships

In [ ]:
import io
from PIL import Image
import pypdfium2 as pdfium
import json
from tqdm import tqdm
from pathlib import Path
import warnings
import pandas as pd
import uuid

def process_row(row, max_pages, base_dir):
    """Process document row into Swift-compatible format"""

    image_format = ".png"
    if "image" in row and "bytes" in row["image"]:
        image = row["image"]
        filename = image['path'] if 'path' in image else uuid.uuid4()
        filetype = "image"
        doc_bytes = image["bytes"]

        
        image_format = image["format"] if "format" in image else ".png"
        image_format = Path(filename).suffix if bool(Path(filename).suffix) else ".png"
    
    else:
        filename = Path(row["filename"]).stem
        filetype = row["filetype"]
        doc_bytes = row["doc_bytes"]

    # Configure image output directory
    images_dir = os.path.join(base_dir, "images")
    os.makedirs(images_dir, exist_ok=True)
    output_images = []

    try:
        if filetype.startswith("image"):
            # Process single image files
            image_path = os.path.join(images_dir, f"{filename}{image_format}")
            if not os.path.exists(image_path):
                Image.open(io.BytesIO(doc_bytes)).save(image_path)
            output_images.append(os.path.relpath(image_path, base_dir))
        
        elif filetype == "application/pdf":
            # Process PDF documents with multi-page support
            pdf = pdfium.PdfDocument(doc_bytes)
            for page_number in range(min(len(pdf), max_pages)):
                page_path = os.path.join(images_dir, f"{filename}_page{page_number:03}{image_format}")
                if not os.path.exists(page_path):
                    pdf[page_number].render(scale=1.53).to_pil().save(page_path)
                output_images.append(os.path.relpath(page_path, base_dir))
    
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
    
    return output_images

def transform_annotations(data, defaults=None, remove_bbox=True):
    """Convert bounding boxes to Swift XML format and simplify structure
    
    Args:
        data: Dictionary or list to transform
        defaults: List of keys that should exist in first-level dictionaries
        remove_bbox: Whether to remove bbox entries from the structure
    """
    bbox_list = []
    bbox_pointer = "<bbox>"
    if isinstance(data, dict):      
        # Add missing default keys (only at the first recursion level)
        if defaults is not None:
            # Use dictionary comprehension for efficiency
            missing_keys = {k: None for k in defaults if k not in data}
            data.update(missing_keys)
        
        for key, value in list(data.items()):            
            if key == "bbox":
                if remove_bbox:
                    del data[key]  # Skip bbox entries when removing
                else:
                    # Convert coordinates from {'bbox': [[20.0, 372.8898], [570.0, 282.8898]]} to {'bbox': [20.0, 372.8898, 570.0, 282.8898]}                    
                    bbox_value = value[0] + value[1]
                    bbox_list.append(bbox_value)
                    data[key] = bbox_pointer
            else:
                value_transformed, bb_list = transform_annotations(value, remove_bbox=remove_bbox)
                bbox_list.extend(bb_list)
                data[key] = value_transformed
        # flatten the object if only one key is left
        if len(data) == 1 and "bbox" not in data:
            data = list(data.values())[0]        
            return data, bbox_list
        return data, bbox_list
        # return {k: v for k, v in data.items() if v is not None}, bbox_list
    elif isinstance(data, list):
        items = []
        for item in data:
            if item is None:
                warnings.warn("Ignoring None value in item list: ", data)
            else:
                item_transformed, bb_list = transform_annotations(item, remove_bbox=remove_bbox)
                items.append(item_transformed)
                bbox_list.extend(bb_list)
        return items, bbox_list
    return data, bbox_list

In [ ]:
# Let's test the transformation on single example entry
row = dataset["test"].to_pandas().iloc[3]
target_format, bbox_list = transform_annotations(json.loads(row[target_column]))
JSON(target_format, expanded=True)

In [ ]:
# lets view the bbox_list, contains items if remove_bbox=False
JSON(bbox_list)

## 5. Amazon Nova Format Conversion

In [ ]:
def collect_all_keys(dataset):
    """Efficiently collect all keys from all dataset splits"""
    print("Collecting all unique keys from all datasets...")
    all_keys = set()
    for split_name, dataset_split in dataset.items():
        df = dataset_split.to_pandas()
        # Extract keys from each target_data JSON and collect unique ones
        df[target_column].apply(json.loads).apply(set).apply(list).explode().drop_duplicates().apply(all_keys.add)
        # df["target_data"].apply(extract_keys).apply(all_keys.update)
    print(f"Found {len(all_keys)} unique keys across all datasets")
    return list(all_keys)

In [ ]:
keys = sorted(collect_all_keys(dataset))

In [ ]:
keys_to_remove = ['AMOUNT_DUE',
     'GST(1%)',
     'GST(12%)',
     'GST(18%)',
     'GST(20%)',
     'GST(5%)',
     'GST(7%)',
     'GST(9%)',
     'GSTIN',
     'GSTIN_BUYER',
     'GSTIN_SELLER',
     'PAYMENT_DETAILS',
     'PO_NUMBER',
     'SELLER_SITE',
     'SEND_TO',
     'LOGO',
     'INVOICE_INFO']

In [ ]:
keys_to_remove = []

In [ ]:
# keys = list(set(keys) - set(keys_to_remove))

In [ ]:
def drop_missing_groundtruth_fields(data):
    
    for key in keys_to_remove:
        data.pop(key, None)
    return data

In [ ]:
from collections import OrderedDict

def order_json(data):
    # Create an OrderedDict with sorted keys
    ordered_data = OrderedDict(sorted(data.items()))
    return ordered_data


In [ ]:
def create_swift_example(row):
    """Construct Swift-compatible training example"""
    conversation = {
        "messages": [
            {
                "role": "system", 
                "content": "You are a document processing expert and assistant."
            },
            {
                "role": "user",
                "content": f"{'Document pages: <image>'*len(row['images'])} Process all document pages and extract the following information in JSON format: {', '.join(keys)}"
            },
            {
                "role": "assistant",
                "content": json.dumps(row["target_data_clean"])
            }
        ],
        "images": row["images"]
        
    }
    
    bbox_list = row["bbox_list"]
    if bbox_list and len(bbox_list):
        conversation["objects"]: {"ref": [], "bbox": bbox_list}
    
    return conversation 

def create_nova_example(row):
    """Construct Swift-compatible training example"""
    user_content = []
    allowed_types = ['png', 'jpg', 'jpeg', 'gif', 'webp']

    for n, image in enumerate(row['images'], start=1):

        ext = Path(image).suffix.lower().strip('.')
        
        if ext in allowed_types:
            user_content.append(
                 {
                    "text": f"Page {n}:"
                }
            )
    
            s3_uri = os.path.join(dataset_s3_uri, image)
            user_content.append(
                {
                    "image": {
                        "format": ext,
                        "source": {
                            "s3Location": {
                                "uri": s3_uri,
                                "bucketOwner": account_id
                            }
                        }
                    }
                }
            )
        else:
            print(f"Warning: Did not add page {n} because image type {ext} is not in allowed types: {allowed_types}. Row: {row}")

    # user_content.append({
    #     "text": "You are a document processing expert and assistant."
    # })
    
    user_content.append({
        "text": f"Process all document pages and extract the following information in JSON format: {', '.join(keys)}"
    })

    
    
    conversation = {
        "schemaVersion": "bedrock-conversation-2024",
        #  TODO Evaluate impact of: Due to the long context tokens of the media file types, the system prompt indicated in the beginning of the prompt might not be respected in certain occasions. 
        #  https://docs.aws.amazon.com/nova/latest/userguide/prompting-vision-prompting.html
        # "system": [{
        #     "text": "You are a document processing expert and assistant."
        # }],
        "messages": [{
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": [{
                    "text": json.dumps(row["target_data_clean"])
                }]
            }
        ]
    }
    
    # bbox_list = row["bbox_list"]
    # if bbox_list and len(bbox_list):
    #     conversation["objects"]: {"ref": [], "bbox": bbox_list}
    
    return conversation 

def convert_dataset(dataset_split, split_name):
    """Full conversion pipeline for dataset split"""
    df = dataset_split.to_pandas()
    
    # Process documents and images
    df["images"] = [process_row(row, 2, dataset_dir) for _, row in tqdm(df.iterrows(), total=len(df))]
    
    df[["target_data_clean","bbox_list"]] = df[target_column].apply(lambda x: pd.Series(transform_annotations(json.loads(x), defaults=keys,remove_bbox=True)))


    # WIP: Removes fields for which the groundtruth data quality is shitty
    # df["target_data_clean"] = df["target_data_clean"].apply(drop_missing_groundtruth_fields)
    
    # Ensure that all attributes are in the same order
    df["target_data_clean"] = df["target_data_clean"].apply(order_json)
    
    # Generate Swift format examples
    converted_data = df.apply(create_nova_example, axis=1) #.tolist()

    # TODO[WIP] data duplication to increase dataset size
    # if split_name in ["train"]:
        # converted_data = converted_data.iloc[converted_data.index.repeat(3)].reset_index(drop=True)
        # converted_data = converted_data.sample(frac=1).reset_index(drop=True)

    
    # Save converted dataset
    output_path = os.path.join(dataset_dir, f"conversations_{split_name}_nova_format.jsonl")
    converted_data.to_json(output_path, orient="records", lines=True)
    # # Write to JSONL file
    # with open(output_path, 'w') as f:
    #     for item in data:
    #         # Convert each item to JSON string and write with newline
    #         f.write(json.dumps(item) + '\n')
    # with open(output_path, "w") as f:
    #     json.dump(converted_data, f, indent=2)
    
    return output_path, df

# Process all dataset splits
swift_files = []
swift_df_all = {}


for split_name, dataset_split in dataset.items():
    print(f"Processing dataset: {split_name}")
    output_file, df = convert_dataset(dataset_split, split_name)
    swift_files.append(output_file)
    swift_df_all[split_name]=df
    print(f"Finished writing: {output_file}")

### Generate and Store JSON Schema format for usage during inference

In [ ]:
# Vertical concatenation
merged = pd.concat([swift_df_all["test"]["target_data_clean"], swift_df_all["train"]["target_data_clean"], swift_df_all["validation"]["target_data_clean"]])

In [ ]:
from genson import SchemaBuilder

# json_targets = swift_df_all["train"]["target_data_clean"]
json_targets = merged
builder = SchemaBuilder()

for item in json_targets:
    builder.add_object(item)

schema = builder.to_schema()

outfile_json_schema = f"{dataset_dir}/groundtruth_schema.json"
#write schema to file
with open(outfile_json_schema, "w") as f:
    json.dump(schema, f, indent=2)
JSON(schema)

In [ ]:
from IPython.display import JSON

# Let's read and show a sample entry of the first dataset
print(f"dataset file: {swift_files[0]}")
data = json.load(open(swift_files[0])) # todo fix json load from jsonl
JSON(data[4], expanded=False, root = "sample") 

In [ ]:
# final target json format, to be generated by the LLM
JSON(json.loads(data[4]["messages"][1]["content"][0]["text"]))

## 6. Upload dataset to S3

In [ ]:
!aws s3 sync $dataset_dir $dataset_s3_uri --exclude ".ipynb_checkpoints/*" --quiet
print(f"\n✅ Dataset successfully uploaded to {dataset_s3_uri}")